# Análisis Exploratorio de Datos

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import pywt

PROJECT_ROOT = Path.cwd().parent
SRC_PATH     = PROJECT_ROOT / "src"
DATA_PATH    = PROJECT_ROOT / "data" / "raw" / "brent_raw.parquet"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from utils import ts_toolkit as tst

df = pd.read_parquet(DATA_PATH)

GROUP_BY_MONTH = {
    "by": lambda idx: idx.month,
    "positions": list(range(1, 13)),
    "labels": [str(m) for m in range(1, 13)],
    "xlabel": "Mes",
    "title": "Distribución mensual",
}

GROUP_BY_WEEKDAY = {
    "by": lambda idx: idx.dayofweek,
    "positions": [0, 1, 2, 3, 4],
    "labels": ["Lun", "Mar", "Mié", "Jue", "Vie"],
    "xlabel": "Día de la semana",
    "title": "Distribución por día de la semana",
}

:::{admonition} Resumen
:class: info
- La serie de precios de cierre presentó no estacionariedad y tendencia; los retornos logarítmicos resultaron estacionarios en media.
- Los retornos logarítmicos mostraron efectos ARCH; los retornos al cuadrado y la volatilidad GK presentaron agrupamiento de volatilidad y persistencia en varianza.
- El período de la pandemia de COVID-19 en 2020 se presentó como el período de mayor volatilidad. El colapso de 2014–2016 y el conflicto en Ucrania de 2022 aparecieron como eventos secundarios.
- Los retornos al cuadrado podrían ser un proxy de volatilidad comparable al estimador GK, especialmente al aplicar ventanas de suavizado.
- Las ventanas de suavizado cortas, de aproximadamente una semana bursátil, filtraron la mayor parte del ruido intradía y revelaron los regímenes persistentes de volatilidad.
:::

## Introducción

El precio del crudo Brent es una variable macroeconómica y financiera global con alta volatilidad y sensibilidad ante eventos geopolíticos. El modelado de varianza condicional en *commodities* evoluciona hacia arquitecturas híbridas que integran series temporales continuas con información no estructurada. La serie histórica de precios del contrato de futuro continuo del Brent proviene de Yahoo Finance. El contrato es un instrumento financiero estandarizado de negociación a plazo. Cotiza en bolsa y sirve de referencia global de precios.

:::{admonition} Información: Dataset de Futuros del Brent
:class: info

La serie analizada proviene del contrato de futuro continuo del petróleo crudo Brent (*ticker BZ=F* en Yahoo Finance). Comprende 3,154 observaciones en días hábiles de mercado (*business days*), cada una con la estructura de precios diarios OHLC (apertura, máximo, mínimo y cierre) y el volumen de negociación.

El período analizado (2014–2026) abarca múltiples regímenes y shocks estructurales. Estos incluyen la crisis de exceso de oferta (2014–2016), las tensiones geopolíticas y sanciones entre Estados Unidos e Irán, el colapso de demanda por COVID‑19 (2020), y el conflicto de Ucrania (2022).
:::

Las series financieras plantean heterocedasticidad condicional, colas pesadas, cambios estructurales de régimen y propagación de shocks exógenos no anticipados. El Análisis Exploratorio de Datos (EDA) constituye una fase crítica de validación e inferencia cuyo propósito es caracterizar la dinámica estocástica de la serie temporal antes de las fases de modelado de la volatilidad.

Esta libreta documenta la evaluación de la integridad de la base de datos: detección de registros faltantes, discontinuidades en el calendario de cotización bursátil y tratamiento de anomalías. También documenta el análisis de raíz unitaria, estacionariedad, propiedades distributivas y estructura de autocorrelación en media y varianza (efectos ARCH). Se incorpora una descomposición multirresolución en el dominio tiempo‑frecuencia para identificar valores atípicos, cambios de régimen y patrones locales de volatilidad.

## Análisis preliminar

El conjunto de datos estuvo compuesto por 3,177 registros diarios correspondientes a días hábiles de mercado. El período abarcó desde el 2 de enero de 2014 hasta el 21 de agosto de 2026. Las cinco variables analizadas (apertura, máximo, mínimo, cierre y volumen) se almacenaron en formato `float32`. No se identificaron valores faltantes en ninguna de ellas.

In [2]:
_ = tst.dataframe_integrity_table(
    df, column_titles=("Variable", "Tipo de dato", "Valores faltantes")
)

Variable,Tipo de dato,Valores faltantes
close,float32,0
high,float32,0
low,float32,0
open,float32,0
volume,float32,0


El conjunto de datos se dividió en dos segmentos: entrenamiento y prueba. El segmento de entrenamiento incluyó 2,223 observaciones, del 2 de enero de 2014 al 3 de noviembre de 2022. El segmento de prueba incluyó 954 observaciones, del 4 de noviembre de 2022 al 21 de agosto de 2026. La proporción fue 70% para entrenamiento y 30% para prueba. El análisis exploratorio se realizó exclusivamente sobre el segmento de entrenamiento. El segmento de prueba no se procesó.

In [4]:
_, _, df_train, df_test = tst.plot_train_test_split(
    df, column="close",
    train_ratio=0.70,
    title="Precio de cierre del petróleo Brent", xlab="Fecha", ylab="Precio (USD/bbl)",
    train_label="Entrenamiento", test_label="Prueba", label_size = 11
)

:::{admonition} Nota: Partición temporal
:class: note
La partición respetó el orden cronológico de los registros. No se aplicó barajado aleatorio. El conjunto de prueba funciona como *holdout set* y permanece intacto para evaluar el desempeño fuera de muestra. Esta estrategia evita la fuga de información entre los conjuntos.
:::

El precio de cierre constituyó la serie principal de referencia. Los retornos logarítmicos diarios se calcularon como $r_t = \ln(P_t/P_{t-1})$. La volatilidad se aproximó mediante dos proxies: los retornos al cuadrado $r_t^2$ y el estimador de rango de Garman‑Klass (GK). El estimador GK utiliza los precios de apertura, máximo, mínimo y cierre para representar la variabilidad intradía. Las series derivadas se calcularon sobre el segmento de entrenamiento.

In [5]:
term_hl = np.log(df_train['high'] / df_train['low']) ** 2
term_co = np.log(df_train['close'] / df_train['open']) ** 2
df_train['gk_daily'] = 0.511 * term_hl - (2 * np.log(2) - 1) * term_co

df_train["log_returns"] = np.log(df_train["close"] / df_train["close"].shift(1))
df_train["squared_returns"] = df_train["log_returns"] ** 2

cols_target = ["close", "log_returns", "squared_returns", "gk_daily"]
df_stats = df_train[cols_target].dropna()

Las estadísticas descriptivas indicaron asimetría positiva en el precio de cierre, con media superior a la mediana. Los retornos logarítmicos mostraron una media cercana a cero y valores extremos que sugieren colas pesadas. Las dos aproximaciones de volatilidad, retornos al cuadrado y estimador Garman‑Klass, presentaron medias y cuartiles próximos, lo que indica consistencia en la captura de la variabilidad diaria. La concentración de la masa en valores bajos frente a máximos elevados apuntan a un agrupamiento de volatilidad.

In [6]:
_ = tst.descriptive_stats_table(
    df_stats,
    column_labels=("Precio de cierre", "Retornos logarítmicos", "Retornos al cuadrado", "Volatilidad GK"),
    param_col_title="Parámetro",
    row_labels=(
        "Registros", "Media", "Desv. Est.", "Mínimo",
        "Cuartil 1", "Mediana", "Cuartil 3", "Máximo", "IQR"
    )
)

Parámetro,Precio de cierre,Retornos logarítmicos,Retornos al cuadrado,Volatilidad GK
Registros,"2,222","2,222","2,222","2,222"
Media,66.4671,-0.0001,0.0007,0.0006
Desv. Est.,21.6286,0.0259,0.0029,0.0022
Mínimo,19.3300,-0.2798,0.0000,0.0000
Cuartil 1,50.3650,-0.0104,0.0000,0.0001
Mediana,62.7350,0.0008,0.0001,0.0003
Cuartil 3,76.4975,0.0115,0.0005,0.0006
Máximo,127.9800,0.1908,0.0783,0.0658
IQR,26.1325,0.0220,0.0005,0.0004


Las estadísticas descriptivas no son representativas por sí solas de la dinámica de precios. No se evaluó la estacionariedad de las series. Una serie no estacionaria presenta media, varianza u otros momentos que dependen del tiempo; los valores descriptivos globales combinan regímenes distintos y no describen un periodo específico. Las próximas secciones presentan un análisis detallado de cada serie, su estacionariedad y posibles eventos atípicos. El EDA culmina con el estudio de la dependencia entre los retornos al cuadrado y el estimador Garman‑Klass para la volatilidad realizada.

:::{admonition} Información: Estimador de Volatilidad de Garman‑Klass
:class: info

El estimador de rango de Garman‑Klass (GK) captura la variabilidad intradía sin depender únicamente de los precios de cierre. Considera la trayectoria diaria de apertura ($O_t$), máximo ($H_t$), mínimo ($L_t$) y cierre ($C_t$):

$$
GK_t = 0.511 \left[\ln\left(\frac{H_t}{L_t}\right)\right]^2 - (2\ln 2 - 1)\left[\ln\left(\frac{C_t}{O_t}\right)\right]^2
$$

El estimador GK logra mayor eficiencia asintótica que la volatilidad basada en retornos de cierre a cierre; esto es: reduce la varianza del estimador al aprovechar la información intradía de precios en comparación con los retornos al cuadrado.
:::

## Precio de cierre

El precio de cierre presentó cuatro episodios de cambio de nivel y volatilidad. El colapso 2014–2016 llevó el precio desde 110 hasta 30 USD/bbl. El auge del petróleo de esquisto en EE.UU. y la defensa de cuotas de mercado por la OPEP estuvieron asociados al descenso. La recuperación 2017–2019 mantuvo el precio entre 50 y 80 USD/bbl. Los acuerdos de recorte OPEP+ respaldaron ese rango. El desplome pandémico de 2020 situó el precio cerca de 20 USD/bbl. El confinamiento global y la parálisis industrial redujeron la demanda. El rally post‑pandemia 2021–2022 superó los 120 USD/bbl. La reactivación económica y las tensiones geopolíticas en Europa del Este impulsaron el alza.

In [7]:
_, _ = tst.plot_series_and_annual_boxplot(
    df_train["close"],
    titles=("Precio de cierre", "Distribución anual del precio"),
    xlab=("Fecha", "Año"), ylab="Precio (USD/bbl)"
)

Los diagramas de caja anuales no fueron homogéneos. La mediana y la amplitud intercuartílica variaron entre años. El año 2014 presentó una distribución asimétrica con valores atípicos en la cola inferior. El año 2016 mostró una caja compacta en precios de cierre más bajos. El año 2020 registró una mediana baja y valores atípicos superiores. El año 2022 desplazó la distribución hacia el segmento superior. Estos patrones sugirieron que la variabilidad del precio no fue constante en el tiempo. La variabilidad fue sensible a eventos macroeconómicos y geopolíticos.

Los diagramas de caja por mes y por día de la semana describieron la distribución del precio. Las medianas mensuales se ubicaron entre 60 y 70 USD/bbl. No se identificó un ciclo estacional definido. Los meses de enero a marzo concentraron valores atípicos en ambas colas. Los movimientos extremos fueron más frecuentes al inicio del año, cuando se concentran mantenimientos de refinerías y decisiones de la OPEP+.

In [7]:
_, _ = tst.plot_ts_boxplots_by_calendar(
    df_train["close"],
    groups=(
        {**GROUP_BY_MONTH,   "title": "Distribución mensual del precio"},
        {**GROUP_BY_WEEKDAY, "title": "Distribución por día de la semana"}
    ),
    ylab="Precio (USD/bbl)"
)

Las distribuciones semanales de los días hábiles (lunes a viernes) mostraron medianas, rangos intercuartílicos y bigotes similares. Esta homogeneidad indicó ausencia de efecto día de la semana en el precio de cierre del Brent.

### Análisis de estacionariedad

La función de autocorrelación (ACF) del precio de cierre mostró un decaimiento lento desde valores cercanos a 1 en el primer rezago. Las autocorrelaciones permanecieron sobre la banda de confianza en los 50 rezagos evaluados. Este patrón sugiere una dependencia temporal fuerte entre el nivel actual y los valores pasados distantes.

In [9]:
_, _ = tst.plot_acf_pacf(
    df_train["close"], titles=["Función de Autocorrelación", "Función de Autocorrelación Parcial"],
    xlab="Rezago", ylab="Autocorrelación"
)

La función de autocorrelación parcial (PACF) mostró un valor alto en el primer rezago y valores cercanos a cero en los rezagos posteriores. El corte después del primer rezago indicó que la dependencia lineal directa se concentró en el primer retardo, una vez controlados los rezagos intermedios.

La serie de precios de cierre no cumplió la condición de estacionariedad, según la prueba Dickey-Fuller aumentada (ADF, *p* = .231) y la prueba Kwiatkowski-Phillips (KPSS, *p* = .010). La serie presentó una raíz unitaria y no revirtió a una media constante. Las pruebas de Ljung‑Box (*p* < .001 en los rezagos 10, 20 y 30) confirmaron la dependencia temporal observada en los gráficos.

In [8]:
_ = tst.univariate_ts_diagnostics(
    df_train["close"], acronym=True,
    p_adjust=(("Ljung-Box", "bonf"),), include_arch=False
)

Prueba,Estadístico,p-valor,p-ajustado,Conclusión (α = .05)
ADF,-2.1353,.231,,No Estacionaria
KPSS,0.8490,.010,,No Estacionaria
Ljung-Box (Lag 10),"21,430.1799",< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),"41,467.1083",< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),"59,936.1725",< .001,< .001,Autocorrelación Significativa


### Descomposición multirresolución

Se aplicó una descomposición wavelet discreta con la wavelet Daubechies 4 (db4) y 7 niveles sobre la serie de precios de cierre del Brent. El componente de tendencia $A_7$ aisló la trayectoria de largo plazo y eliminó las fluctuaciones de alta frecuencia. Este componente reprodujo los episodios identificados en la inspección visual del precio y sirvió como referencia del nivel subyacente del mercado. Los cambios de régimen se manifestaron como desplazamientos persistentes en la media de la serie, no como oscilaciones de corto plazo.

:::{admonition} Nota: Descomposición Wavelet con db4
:class: note

La descomposición wavelet con db4 representa la serie de precios como la suma de una aproximación de baja frecuencia y varios detalles de alta frecuencia. La wavelet db4 posee soporte compacto y dos momentos de desvanecimiento. El soporte compacto permite localizar cambios en el tiempo, y los momentos de desvanecimiento permiten representar tendencias suaves con pocos coeficientes.

$$
P_t = A_J(t) + \sum_{j=1}^{J} D_j(t)
$$

Donde $A_J$ es la aproximación final y $D_j$ son los detalles en cada escala. La descomposición wavelet no requiere supuestos sobre periodicidad fija. Esta característica la diferencia de la descomposición STL (*Seasonal-Trend decomposition using Loess*), que asume estacionalidad fija, y de los modelos aditivos o multiplicativos tradicionales, que imponen una estructura rígida a la estacionalidad. La flexibilidad de la wavelet db4 la hace adecuada para estudiar la dinámica del precio del Brent.
:::

El componente de detalle $D_7$ osciló alrededor de cero y capturó fluctuaciones de frecuencia intermedia. La amplitud de las oscilaciones no fue constante: aumentó en períodos de tensión como 2014–2015 y el inicio de 2020, y se atenuó en intervalos de menor volatilidad como 2016–2017 y parte de 2021. Los ciclos de mediano plazo no siguieron un patrón fijo de calendario; se modularon según las condiciones del mercado. Se amplificaron durante shocks macroeconómicos y se contrajeron en fases de relativa estabilidad. La variación de la amplitud sugirió que la volatilidad de los retornos dependía de la escala del ciclo.

In [11]:
close_prices = df_train["close"].values
wavelet = "db4"
level = 7
coeffs = pywt.wavedec(close_prices, wavelet, level=level)

_, _ = tst.plot_wavelet_decomposition(
    coeffs=coeffs, index=df_train.index, n_obs=len(close_prices),
    wavelet=wavelet, levels=(0, 1),
    panel_titles=("Componente de tendencia", "Estacionalidad anualizada"),
    ylab=("Precio de cierre (USD/bbl)", "Desviación"), xlab="Fecha"
)

El componente semestral $D_6$ mostró un aumento de amplitud desde finales de 2018. Las oscilaciones se mantuvieron en un rango aproximado de ±5 USD/bbl antes de ese período; después, los picos alcanzaron valores cercanos a ±10 USD/bbl. La expansión coincidió con la recuperación del precio tras la depresión de 2014–2016 y con los recortes de producción de la OPEP+ de finales de 2018. La energía cíclica de mediano plazo se incrementó cuando el precio se situó en niveles más altos.

La banda trimestral $D_4 + D_5$ mostró sensibilidad a crisis puntuales. Las oscilaciones se comprimieron en períodos de calma y se expandieron en pulsos de alta frecuencia durante la guerra de precios de 2015, el colapso de demanda por la COVID‑19 en 2020 y la dislocación energética de 2022. La banda concentró su energía en picos localizados y no presentó un cambio de régimen sostenido, a diferencia del componente semestral. La naturaleza transitoria de los shocks quedó reflejada en esos pulsos.

In [16]:
_, _ = tst.plot_wavelet_decomposition(
    coeffs=coeffs, index=df_train.index, n_obs=len(close_prices),
    wavelet=wavelet, levels=(2, (3, 4)),
    panel_titles=("Estacionalidad semestral", "Estacionalidad trimestral"),
    ylab="Desviación", xlab="Fecha"
)

El componente irregular ($D_1+D_2+D_3$) agrupó oscilaciones de período inferior a un mes. Su amplitud se mantuvo en un rango de ±6 USD/bbl antes de 2022, con picos ocasionales durante las crisis de 2015 y 2020. A partir de 2022, las oscilaciones superaron ±10 USD/bbl y la amplitud permaneció elevada. Ese aumento coincidió con la invasión de Ucrania y las sanciones energéticas posteriores, que introdujeron incertidumbre diaria sobre la oferta física de crudo.

In [13]:
_, _ = tst.plot_wavelet_decomposition(
    coeffs=coeffs, index=df_train.index, n_obs=len(close_prices),
    wavelet=wavelet, levels=((5, 6, 7),),
    panel_titles=("Variaciones irregulares",), 
    ylab="Desviación", xlab="Fecha",
    title_loc="center",
    yticks=[-10, -5, 0, 5, 10], ylim=(-13, 13),
)

El aumento de la varianza de alta frecuencia no se reflejó con la misma intensidad en la tendencia ni en los ciclos semestrales o trimestrales. La tendencia $A_7$ mostró una senda alcista desde 2020. Las fluctuaciones diarias del precio se amplificaron sin modificar la dirección de fondo y se desacoplaron de la trayectoria de largo plazo. Los shocks de corto plazo se concentraron en las variaciones intrasemanales e intradías. Los resultados sugirieron que los operadores reaccionaron ante titulares diarios (inventarios, declaraciones de la OPEP, sanciones), lo que habría inyectado mayor energía en la alta frecuencia durante períodos de tensión geopolítica o escasez física, en especial después de 2022.

La descomposición wavelet mostró que los componentes estacionales no mantuvieron amplitudes constantes: se amplificaron en períodos de tensión y se atenuaron en fases de calma. La falta de estacionariedad y la presencia de ruido pudieron afectar la potencia wavelet y la detección de eventos atípicos. Los cambios de nivel elevaron la potencia en escalas largas y pudieron generar detecciones asociadas a desplazamientos del precio, no a perturbaciones locales. El ruido transitorio pudo superar el umbral en momentos aislados, sin una interpretación estructural clara. La ausencia de estacionariedad y la variabilidad de los patrones estacionales limitaron el análisis directo de la serie mediante métodos clásicos de series de tiempo.

## Retornos logarítmicos

La serie de precios de cierre presentó no estacionariedad y tendencia, lo que limitó el análisis directo. Los retornos logarítmicos se calcularon para transformar la serie y proporcionar una base adecuada para estudiar la volatilidad. Esta transformación estabiliza la media y reduce la dependencia del nivel de precios.

:::{admonition} Nota: Retorno Logarítmico y Diferenciación
:class: note

La diferenciación de una serie temporal $P_t$ produce una nueva serie $\Delta P_t = P_t - P_{t-1}$. Esta operación elimina tendencias lineales y ayuda a inducir estacionariedad en media. En series de precios financieros, la diferenciación simple no resulta adecuada porque la variabilidad suele depender del nivel del precio. El retorno logarítmico resuelve esa limitación:

$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right) = \ln(P_t) - \ln(P_{t-1})
$$

El retorno logarítmico corresponde a la primera diferencia del logaritmo del precio. Esta transformación estabiliza la varianza y se interpreta como la variación porcentual continua del precio entre dos períodos consecutivos.
:::

La serie de retornos logarítmicos se mantuvo centrada en cero. La media anual permaneció cercana a cero y las cajas anuales no mostraron desplazamientos de nivel. Los diagramas de caja revelaron valores atípicos en ambas colas. El año 2020 registró retornos negativos cercanos a -0.28. Estos resultados sugirieron la presencia de colas pesadas en la distribución de retornos.

In [18]:
_, _ = tst.plot_series_and_annual_boxplot(
    df_train["log_returns"],
    titles=("Retornos del Brent", "Distribución anual de retornos"),
    xlab=("Fecha", "Año"), ylab="Retorno logarítmico",
    ylim=(-0.3, 0.3), y_step=0.1, y_format="%.2f"
)

Los diagramas de caja de los retornos logarítmicos mostraron una concentración de valores atípicos en los meses de enero a marzo, un patrón similar al observado en la serie de precios de cierre. Abril se incorporó a ese bloque de alta dispersión. Los retornos negativos extremos aparecieron con mayor frecuencia en el primer cuatrimestre del año. La inclusión de abril sugirió que su variabilidad diaria fue comparable a la de los primeros meses del año, a diferencia del nivel de precios, que no registró extremos comparables. Los resultados sugirieron que la transición estacional y el inicio de la temporada de mayor demanda de combustibles podrían estar asociados con esa mayor dispersión.

In [14]:
_, _ = tst.plot_ts_boxplots_by_calendar(
    df_train["log_returns"],
     groups=(
        {**GROUP_BY_MONTH,   "title": "Distribución mensual de retornos"},
        {**GROUP_BY_WEEKDAY, "title": "Distribución semanal de retornos"}
    ),
    ylab="Retorno logarítmico", ylim=(-0.3, 0.3), sharey=True
)

La distribución por día de la semana mostró diferencias en los retornos logarítmicos que no se observaron en el precio de cierre. La serie de precios no presentó diferencias marcadas entre días hábiles. Lunes y martes registraron valores atípicos inferiores cercanos a -0.28 en los retornos, mientras los demás días presentaron una distribución más acotada. Estos resultados sugirieron que la publicación de inventarios semanales y la reacción a noticias del fin de semana podrían estar asociadas con la mayor dispersión observada al inicio de la semana.

### Análisis de estacionariedad

La ACF de los retornos logarítmicos se mantuvo dentro de la banda de confianza en los 50 rezagos evaluados. La función de autocorrelación parcial PACF mostró un comportamiento similar: ningún rezago individual superó la banda. Este patrón contrastó con la serie de precios de cierre, cuya ACF presentó decaimiento lento y persistencia sobre la banda. Los retornos logarítmicos no mostraron dependencia lineal; el comportamiento fue compatible con ruido blanco en media.

In [20]:
_, _ = tst.plot_acf_pacf(
    df_train["log_returns"], titles=["Función de Autocorrelación", "Función de Autocorrelación Parcial"],
    xlab="Rezago", ylab="Autocorrelación"
)

Las pruebas ADF (*p* < .001) y KPSS (*p* = .100) respaldaron la estacionariedad de los retornos logarítmicos. Las pruebas de Ljung‑Box indicaron autocorrelación significativa en los rezagos 20 y 30 (*p* < .001); el rezago 10 no alcanzó significación (*p* = .057). La autocorrelación en rezagos largos sugirió una dependencia temporal no capturada por la ACF visual, posiblemente asociada al agrupamiento de volatilidad.

:::{admonition} Nota: Ruido Blanco
:class: note

En series económicas y financieras, el concepto de ruido blanco admite dos niveles. Una serie $y_t$ es ruido blanco débil si cumple:

$$
\mathbb{E}[y_t] = \mu,\qquad \mathbb{E}[(y_t-\mu)(y_{t-k}-\mu)] = 0 \quad \, \forall k \neq 0.
$$

La media es constante y no existe autocorrelación lineal. La varianza incondicional puede ser constante, pero no se exige independencia en varianza.

Una serie $y_t$ es ruido blanco fuerte si, además de las condiciones anteriores, las observaciones son independientes e idénticamente distribuidas (i.i.d.). En este caso, no existe dependencia lineal ni no lineal, y la varianza condicional coincide con la varianza incondicional.
:::

Las pruebas ARCH‑LM indicaron presencia de efectos ARCH en los tres rezagos (*p* < .001). Este resultado sugirió que la varianza condicional no es constante y que los retornos presentan agrupamiento de volatilidad. La evidencia no invalidó la hipótesis de ruido blanco en media, pero señaló que la independencia en varianza no se cumple. Los retornos serían compatibles con un ruido blanco débil, no con un ruido blanco fuerte.

In [15]:
_ = tst.univariate_ts_diagnostics(
    df_train["log_returns"], acronym=True,
    p_adjust=(("Ljung-Box", "bonf"), ("ARCH-LM", "bonf"))
)

Prueba,Estadístico,p-valor,p-ajustado,Conclusión (α = .05)
ADF,-9.4693,< .001,,Estacionaria
KPSS,0.2108,.100,,Estacionaria
Ljung-Box (Lag 10),17.8667,.057,.172,Sin Autocorrelación Significativa
Ljung-Box (Lag 20),49.0930,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),73.2864,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),217.9392,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),320.3613,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),479.3182,< .001,< .001,Efectos ARCH Significativos


### Descomposición multirresolución

La descomposición wavelet db4 se aplicó a la serie de retornos logarítmicos con el mismo nivel de descomposición que en los precios de cierre. El objetivo fue comparar la dinámica de la volatilidad en diferentes escalas temporales.

La tendencia estimada de los retornos logarítmicos mostró una caída desde 2021, en contraste con la senda alcista de los precios de cierre. Los valores pasaron de niveles cercanos a cero a registros negativos durante el conflicto en Ucrania de 2022. Este resultado sugirió que el impulso alcista del precio perdió fuerza en términos de tasa de cambio antes de alcanzar su máximo nominal. La desaceleración relativa de los retornos indicó un debilitamiento estructural del crecimiento del precio.

El análisis de la escala anual reveló oscilaciones atenuadas en los retornos durante el colapso de 2014–2015, mientras que la serie de precios presentó una oscilación amplia y suave en ese mismo período. La energía se concentró en la pandemia de COVID-19 de 2020, con un pico de amplitud superior al de cualquier otro año. Este patrón sugirió que el colapso de 2014–2015 fue una deriva de nivel progresiva, mientras que la pandemia generó una perturbación de alta frecuencia en los retornos.

In [23]:
df_plot = df_train.dropna(subset=["log_returns"]).copy()
retornos = df_plot["log_returns"].values
n_obs = len(retornos)

wavelet = "db4"
level = 7
coeffs = pywt.wavedec(retornos, wavelet, level=level)

_, _ = tst.plot_wavelet_decomposition(
    coeffs=coeffs, index=df_plot.index, n_obs=n_obs,
    wavelet=wavelet, levels=(0, 1),
    panel_titles=("Componente de tendencia", "Estacionalidad anualizada"),
    ylab=("Retorno logarítmico", "Desviación"), xlab="Fecha"
)

Las oscilaciones semestrales de los retornos mostraron picos de amplitud durante el colapso de 2014–2015 y las tensiones geopolíticas de 2018, con valores comparables a los de la pandemia de 2020. En la serie de precios, esos períodos presentaron caídas graduales. La amplificación en retornos sugirió que la variación porcentual acumulada generó mayor energía en esa escala. Durante el conflicto en Ucrania de 2022, la amplitud en retornos fue menor que en precios, lo que reflejó un efecto de escala: oscilaciones absolutas grandes sobre un nivel de precio alto representaron variaciones porcentuales moderadas.

La comparación entre retornos y precios en la banda trimestral mostró una compresión en los retornos durante el colapso de 2015 y el conflicto en Ucrania de 2022, con valores cercanos al ruido base. La serie de precios presentó picos de amplitud en esos mismos períodos. La pandemia de COVID-19 de 2020 concentró la energía de esta banda, con una amplitud varias veces superior a la de otros eventos. Este resultado sugirió que el colapso de 2015 y el conflicto en Ucrania fueron fluctuaciones de baja frecuencia impulsadas por el nivel de precio, mientras que la pandemia constituyó un choque de alta frecuencia.

In [24]:
_, _ = tst.plot_wavelet_decomposition(
    coeffs=coeffs, index=df_plot.index, n_obs=n_obs,
    wavelet=wavelet, levels=(2, (3, 4)),
    panel_titles=("Estacionalidad semestral", "Estacionalidad trimestral"),
    ylab="Desviación", xlab="Fecha"
)

La banda de alta frecuencia de los retornos mostró una compresión durante el conflicto en Ucrania de 2022 y una amplificación durante la pandemia de COVID-19 de 2020. La serie de precios presentó las mayores desviaciones nominales en el conflicto de 2022, mientras que la pandemia mostró picos menores. Este contraste sugirió un efecto de escala: las variaciones absolutas grandes de 2022 representaron cambios porcentuales moderados, mientras que las variaciones de 2020, menores en dólares, fueron extremas en términos porcentuales. Los resultados indicaron que la pandemia fue un episodio de hipervolatilidad de alta frecuencia, y el conflicto en Ucrania, un período de incertidumbre de nivel.

In [25]:
max_abs = float(max(
    abs(pywt.waverec(
        [np.zeros_like(c) if i not in (5, 6, 7) else c for i, c in enumerate(coeffs)],
        wavelet,
    )[:n_obs].min()),
    abs(pywt.waverec(
        [np.zeros_like(c) if i not in (5, 6, 7) else c for i, c in enumerate(coeffs)],
        wavelet,
    )[:n_obs].max()),
))

_, _ = tst.plot_wavelet_decomposition(
    coeffs=coeffs, index=df_plot.index, n_obs=n_obs,
    wavelet=wavelet, levels=((5, 6, 7),),
    panel_titles=("Variaciones irregulares",),
    ylab="Desviación", xlab="Fecha",
    title_loc="center",
    ylim=(-max_abs * 1.1, max_abs * 1.1),
)

La diferenciación logarítmica eliminó el efecto de escala de los precios nominales. Eventos como el colapso de 2014–2016 y el conflicto en Ucrania en 2022 mostraron amplitudes moderadas en los retornos: el primero por su carácter de deriva de nivel y el segundo por representar cambios porcentuales reducidos sobre un nivel de precio alto.

La descomposición wavelet de los retornos destacó el shock de la pandemia de COVID-19 en 2020 como un período de hipervolatilidad porcentual. Los resultados sugirieron que la pandemia fue una perturbación de alta frecuencia sin equivalente en el resto de la serie, mientras que el colapso de 2014–2016 y el conflicto en Ucrania constituyeron episodios de incertidumbre de nivel con menor variabilidad relativa.

## Retornos al cuadrado

Los retornos al cuadrado $\left(r^2\right)$ se calcularon a partir de los retornos logarítmicos para obtener una aproximación directa de la varianza diaria no observable. Esta transformación elimina el signo de los retornos y conserva la magnitud de las variaciones, lo que permite estudiar la volatilidad realizada.

La serie de $r^2$ mostró una agrupación temporal de la varianza: los períodos de alta volatilidad tendieron a sucederse. Los picos de mayor magnitud se concentraron en eventos de tensión, con la pandemia de COVID-19 en 2020 como el episodio más extremo. La transformación acentuó los eventos ya observados en la serie de retornos logarítmicos. Los diagramas de caja anuales indicaron un régimen base estable, con medianas cercanas a cero en todos los años, pero revelaron valores extremos en la cola derecha durante 2020. Este patrón sugirió que la crisis de 2020 no fue un cambio sostenido en el nivel medio de la volatilidad diaria, sino un shock de cola de alta frecuencia.

In [26]:
_, _ = tst.plot_series_and_annual_boxplot(
    df_train["squared_returns"],
    titles=(r"Retornos al cuadrado ($r^2$)", r"Distribución anual de $r^2$"),
    xlab=("Fecha", "Año"), ylab="Retorno al cuadrado"
)

Los diagramas de caja mensuales de $r^2$ mostraron un nivel base homogéneo entre los doce meses, con medianas cercanas a cero. Los valores infrecuentes se concentraron en marzo y abril, meses que registraron las magnitudes más altas de la muestra. Este patrón coincidió con el inicio de los cierres globales por COVID-19 y la ruptura de cuotas OPEP+ en la primavera de 2020. Septiembre y noviembre presentaron picos secundarios, mientras que junio y octubre fueron los meses más estables. A diferencia de los retornos logarítmicos, donde los extremos se ubicaron en el primer trimestre y se asociaron a retornos negativos, los retornos al cuadrado concentraron su dispersión en los meses de mayor turbulencia de 2020, sin desplazar la mediana del nivel base.

In [16]:
_, _ = tst.plot_ts_boxplots_by_calendar(
    df_train["squared_returns"],
    groups=(
        {**GROUP_BY_MONTH,   "title": r"Distribución mensual de $r^2$"},
        {**GROUP_BY_WEEKDAY, "title": r"Distribución semanal de $r^2$"}
    ),
    ylab="Retorno al cuadrado",
    ylim=(-0.001, 0.06),
    sharey=True,
)

La distribución por día de la semana de $r^2$ mostró medianas y cuartiles aparentemente similares entre lunes y viernes. Los valores infrecuentes se concentraron en jueves, con el punto más extremo de la muestra, y en miércoles y viernes, con una mayor densidad de picos intermedios. Este comportamiento contrastó con el de los retornos logarítmicos, donde lunes y martes registraron los valores infrecuentes inferiores. Los resultados sugirieron que los saltos de volatilidad tendieron a ubicarse hacia la segunda mitad de la semana hábil, lo que podría estar asociado con la publicación de inventarios de crudo en Estados Unidos y el ajuste de posiciones previo al fin de semana.

### Análisis de estacionariedad

La ACF de los retornos al cuadrado reveló autocorrelaciones significativas en la mayoría de los rezagos del 1 al 35, con valores moderados que oscilaron entre 0.10 y 0.25. Este patrón sugirió una persistencia en la varianza que no se desvaneció rápidamente, a diferencia de lo esperado para una serie sin memoria. Un pico pronunciado cerca del rezago 30 alcanzó aproximadamente 0.35, lo que indicó una posible dependencia cíclica mensual. La presencia de múltiples rezagos fuera de la banda de confianza respaldó la hipótesis de memoria larga en la volatilidad.

In [28]:
_, _ = tst.plot_acf_pacf(
    df_train["squared_returns"], titles=["Función de Autocorrelación", "Función de Autocorrelación Parcial"],
    xlab="Rezago", ylab="Autocorrelación"
)

El análisis de la PACF identificó picos de dependencia en los primeros rezagos (1, 3, 7, 10 y 12), con valores entre 0.10 y 0.20. El rezago 30 mantuvo un pico dominante cercano a 0.30, lo que sugirió que la relación mensual no era un efecto propagado, sino una componente directa. A partir del rezago 35, los valores ingresaron dentro de la banda de confianza, indicando un decaimiento en los órdenes superiores.

:::{admonition} Información: ACF y PACF sobre retornos al cuadrado
:class: info

En econometría de series temporales y finanzas cuantitativas, la ACF y la PACF sobre los retornos al cuadrado no buscan modelar la media, sino verificar la persistencia en la varianza condicional (efectos ARCH/GARCH). 

Los efectos ARCH (*Autoregressive Conditional Heteroskedasticity*) se refieren a la dependencia temporal de la varianza: la volatilidad en un período está correlacionada con la de períodos anteriores.
:::

Las pruebas de Ljung‑Box rechazaron la ausencia de autocorrelación en los rezagos 10, 20 y 30 (*p* < .001). Estos resultados confirmaron la dependencia temporal observada previamente y respaldaron la hipótesis de efectos ARCH. La prueba ADF (*p* < .001) y la prueba KPSS (*p* = .085) indicaron que la serie de retornos al cuadrado no presentó raíz unitaria y mantuvo un comportamiento estacionario en media.

In [17]:
_ = tst.univariate_ts_diagnostics(
    df_train["squared_returns"], acronym=True,
    p_adjust=(("Ljung-Box", "bonf"),), include_arch=False
)

Prueba,Estadístico,p-valor,p-ajustado,Conclusión (α = .05)
ADF,-5.5771,< .001,,Estacionaria
KPSS,0.3825,.085,,Estacionaria
Ljung-Box (Lag 10),371.1136,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),749.7437,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),"1,163.8521",< .001,< .001,Autocorrelación Significativa


### Descomposición multirresolución

La tendencia de los retornos al cuadrado mostró una meseta moderada durante el colapso de 2014–2016, sin niveles explosivos. La pandemia de COVID-19 en 2020 produjo una cúspide dominante, muy superior a cualquier otro período. El conflicto en Ucrania de 2022 elevó ligeramente la varianza, pero se mantuvo lejos de la magnitud de 2020. Este contraste indicó que la pandemia concentró la varianza estructural, mientras que los eventos de 2014–2016 y 2022 tuvieron un efecto más limitado.

La escala anual de los retornos al cuadrado mostró oscilaciones durante 2015–2016 y 2022, aunque con amplitudes menores que las de 2020. La pandemia generó una estructura de triple pico con valles negativos, lo que sugirió una secuencia de choques de volatilidad durante el año. Este patrón indicó que la varianza anualizada se concentró de manera asimétrica durante la pandemia, mientras que los otros eventos produjeron fluctuaciones más atenuadas.

In [31]:
df_plot = df_train.dropna(subset=["squared_returns"]).copy()
retornos = df_plot["squared_returns"].values
n_obs = len(retornos)

wavelet = "db4"
level = 7
coeffsr = pywt.wavedec(retornos, wavelet, level=level)

_, _ = tst.plot_wavelet_decomposition(
    coeffs=coeffsr, index=df_plot.index, n_obs=n_obs,
    wavelet=wavelet, levels=(0, 1),
    panel_titles=("Componente de tendencia", "Estacionalidad anualizada"),
    ylab=("Retorno al cuadrado", "Desviación"), xlab="Fecha"
)

El componente semestral de los retornos al cuadrado replicó la morfología del componente anualizado, con menor amplitud y mayor frecuencia. Fuera de los períodos de crisis, la escala semestral permaneció plana, en contraste con los retornos logarítmicos, que mostraron oscilaciones continuas. La pandemia de 2020 generó una doble cresta con un valle intermedio, mientras que en 2022 apareció una cresta secundaria de menor amplitud. Este patrón sugirió que la energía de la varianza semestral se concentró en 2020, y que los demás eventos tuvieron un efecto limitado en esa escala.

:::{admonition} Nota: Transformación cuadrática de los retornos
:class: note

La transformación cuadrática

$$
r_t \to r_t^2
$$

atenúa las variaciones pequeñas y amplifica los saltos extremos. Un retorno de 0.01 se convierte en 0.0001, mientras que uno de 0.20 se convierte en 0.04. Este efecto reduce la contribución de la volatilidad moderada y hace que los episodios de alta magnitud dominen la serie resultante.
:::

La escala trimestral de los retornos al cuadrado también reprodujo la estructura observada en los componentes anual y semestral, comprimida en un marco temporal más corto. Los retornos logarítmicos, en cambio, presentaron una envolvente más ruidosa con actividad en 2015 y 2022. La transformación cuadrática atenuó las variaciones pequeñas y amplificó los saltos extremos, lo que aisló el shock de 2020 y redujo las oscilaciones de otros períodos a valores residuales. Los resultados indicaron que la pandemia activó de forma simultánea varias escalas temporales de la varianza, con una firma morfológica similar en los tres niveles.

In [32]:
_, _ = tst.plot_wavelet_decomposition(
    coeffs=coeffsr, index=df_plot.index, n_obs=n_obs,
    wavelet=wavelet, levels=(2, (3, 4)),
    panel_titles=("Estacionalidad semestral", "Estacionalidad trimestral"),
    ylab="Desviación", xlab="Fecha"
)


El componente de alta frecuencia de los retornos al cuadrado mostró una compresión del ruido base respecto a los retornos logarítmicos. La banda densa y continua que estos presentaban se redujo a una línea casi plana, debido a la atenuación de las fluctuaciones moderadas. Los picos de 2015–2016 y 2022 permanecieron perceptibles, con amplitudes cercanas a 0.010 y 0.015, respectivamente. El período de la pandemia de 2020 dominó la componente con un impulso agudo que alcanzó 0.070.

In [33]:
_, _ = tst.plot_wavelet_decomposition(
    coeffs=coeffsr, index=df_plot.index, n_obs=n_obs,
    wavelet=wavelet, levels=((5, 6, 7),),
    panel_titles=("Variaciones irregulares",),
    ylab="Desviación", xlab="Fecha",
    title_loc="center", ylim=(-0.1, 0.1)
)

### Suavizado por media móvil

Los retornos al cuadrado constituyen una medida ruidosa de la varianza diaria, sensible a fluctuaciones transitorias de alta frecuencia. El suavizado mediante medias móviles reduce ese ruido y permite observar la varianza condicional en escalas semanales, quincenales y mensuales.

:::{admonition} Nota: Media móvil centrada
:class: note

El suavizado por media móvil centrada de ventana $w$ se define como:

$$
\tilde{r}^2_t = \frac{1}{w} \sum_{i=-(w-1)/2}^{(w-1)/2} r^2_{t+i}
$$

para ventanas impares, o como el promedio de los $w$ valores centrados alrededor de $t$ cuando $w$ es par. La ventana de 5 días corresponde a una semana bursátil, la de 10 días a dos semanas y la de 21 días a un mes hábil.
:::

La serie sin suavizar mostró picos agudos, con el período de la pandemia de 2020 alcanzando valores cercanos a 0.08. La ventana de 5 días filtró parte de la variabilidad intradía y redujo ese máximo a 0.02. Este suavizado reveló estructuras de volatilidad en 2015–2016 y 2022, con amplitudes entre 0.003 y 0.005. 

La ventana de 10 días atenuó aún más los picos y perfiló dos jorobas en 2014–2016, una meseta ancha en 2020 y una elevación moderada en 2022. Con 21 días, la serie se transformó en una curva suave donde se distinguieron tres regímenes de varianza elevada: 2015–2016, 2020 y 2022. Los resultados indicaron que el suavizado separó los saltos de corta duración de los estados persistentes de volatilidad.

In [18]:
df_plot = df_train.dropna(subset=["squared_returns"]).copy()

_, _ = tst.plot_rolling_smoothing_panels(
    df_plot["squared_returns"],
    titles=(
        "Retornos al cuadrado",
        "Retornos al cuadrado (suavizado 5 días)",
        "Retornos al cuadrado (suavizado 10 días)",
        "Retornos al cuadrado (suavizado 21 días)"
    ),
    ylab="Retorno al cuadrado", xlab="Fecha",
    ylim=(None, (-0.001, 0.03), (-0.001, 0.02), (-0.001, 0.02)),
    y_format=("%.3f", None, None, None),
    yticks=(None, np.arange(0.002, 0.029, 0.006), None, None),
    tick_size=10, hspace=0.30
)

## Volatibilidad GK

La serie de volatilidad GK mostró una morfología similar a la de los retornos al cuadrado. El colapso del precio del crudo en 2014–2016, la pandemia de COVID-19 en 2020 y el conflicto en Ucrania en 2022 también se observaron en esta medida, con picos de menor magnitud que los de $r^2$. El período de la pandemia concentró la mayor variabilidad, aunque con valores máximos inferiores a los de los retornos al cuadrado. Este patrón sugirió que ambas medidas capturaron los mismos episodios de tensión, pero con escalas diferentes.

:::{admonition} Información: Estimadores de rango de la volatilidad
:class: info

Existen varios estimadores de volatilidad basados en el rango, como Parkinson, Rogers‑Satchell o Yang‑Zhang. La elección del estimador Garman‑Klass respondió a la información disponible en el conjunto de datos, que incluye precios de apertura, máximo, mínimo y cierre. 

El estimador Yang‑Zhang es más eficiente que el GK: alcanza una eficiencia relativa cercana a 8 veces la varianza de cierre a cierre, mientras que el GK se sitúa en torno a 7.4 y Rogers‑Satchell en 6.0. 

A pesar de no ser el más eficiente, el GK constituye una alternativa estándar en la literatura y un punto de comparación válido frente a los retornos al cuadrado.
:::

Los diagramas de caja anuales de la volatilidad GK revelaron un menor número de valores extremos en comparación con los retornos al cuadrado. La mayoría de los años presentaron rangos intercuartílicos reducidos y bigotes acotados, lo que sugirió una variabilidad diaria más estable. Los valores más alejados se concentraron en 2020, con magnitudes inferiores a las observadas en $r^2$. Este comportamiento indicó que la GK, al incorporar información intradía, suavizó parte de la variabilidad extrema capturada por los retornos al cuadrado.

In [36]:
_, _ = tst.plot_series_and_annual_boxplot(
    df_train["gk_daily"],
    titles=("Volatilidad GK diaria", "Distribución anual de la volatilidad"),
    xlab=("Fecha", "Año"), ylab="Volatilidad GK"
)

La distribución mensual de la volatilidad GK mostró valores de mayor magnitud en marzo y abril, un patrón similar al observado en los retornos al cuadrado. Los picos de la GK fueron menores que los de $r^2$, pero se concentraron en los mismos meses. Esta coincidencia reforzó la idea de que ambos estimadores capturaron los episodios de tensión de la primavera de 2020, cuando se combinaron la ruptura de cuotas OPEP+ y el inicio de la pandemia.

In [19]:
_, _ = tst.plot_ts_boxplots_by_calendar(
    df_train["gk_daily"],
    groups=(
        {**GROUP_BY_MONTH,   "title": "Distribución mensual de volatilidad"},
        {**GROUP_BY_WEEKDAY, "title": "Distribución semanal de volatilidad"}
    ),
    ylab="Volatilidad GK", ylim=(-0.001, 0.07), sharey=True
)

La distribución por día de la semana mostró un desplazamiento de los valores más extremos de jueves a miércoles. En los retornos al cuadrado, los valores infrecuentes se concentraban en jueves. La GK, en cambio, los ubicó en miércoles. Este cambio podría estar asociado con la publicación de inventarios de crudo en Estados Unidos, que ocurre los miércoles. La GK, al incorporar la trayectoria intradía, capturó la reacción a ese anuncio el mismo día, mientras que los retornos al cuadrado la reflejaron con mayor intensidad en el cierre del día siguiente.

### Análisis de estacionariedad

La comparación de las funciones de autocorrelación reveló diferencias marcadas entre la volatilidad GK y los retornos al cuadrado. La ACF de los retornos al cuadrado mostró una persistencia moderada, con un primer rezago cercano a 0.20 y un pico aislado en el rezago 30. La ACF de la volatilidad GK, en cambio, presentó un primer rezago superior a 0.60 y mantuvo valores por encima de 0.20 durante los primeros treinta rezagos, con un decaimiento lento. Este patrón sugirió que la GK capturó una estructura de memoria larga en la varianza condicional, mientras que los retornos al cuadrado conservaron una mayor proporción de ruido.

In [41]:
_, _ = tst.plot_acf_pacf(
    df_train["gk_daily"], titles=["Función de Autocorrelación", "Función de Autocorrelación Parcial"],
    xlab="Rezago", ylab="Autocorrelación"
)

La PACF de los retornos al cuadrado mostró picos dispersos en los rezagos 1, 3, 7, 10 y 12, junto con una espiga en el rezago 30. La PACF de la GK, por su parte, concentró la información autorregresiva en el primer rezago, con un coeficiente cercano a 0.60, y presentó un segundo rezago negativo de menor magnitud. El pico en el rezago 30 se redujo de forma considerable en la GK, lo que sugirió que la aparente estacionalidad mensual observada en los retornos al cuadrado estaba influida por la varianza residual de los cierres diarios. La mayor eficiencia del estimador GK, al incorporar la trayectoria intradía de precios, redujo el ruido muestral y permitió identificar una estructura de dependencia más definida y persistente.

Las pruebas de Ljung‑Box rechazaron la ausencia de autocorrelación en los rezagos 10, 20 y 30 (*p* < .001). Estos resultados coincidieron con la estructura de autocorrelación observada en los gráficos y respaldaron la presencia de dependencia temporal en la varianza. Las pruebas ADF (*p* < .001) y KPSS (*p* = .065) indicaron que la serie GK mantuvo un comportamiento estacionario en media.

In [22]:
_ = tst.univariate_ts_diagnostics(
    df_train["gk_daily"], acronym=True,
    p_adjust=(("Ljung-Box", "bonf"),), include_arch=False
)

Prueba,Estadístico,p-valor,p-ajustado,Conclusión (α = .05)
ADF,-5.4364,< .001,,Estacionaria
KPSS,0.4274,.065,,Estacionaria
Ljung-Box (Lag 10),"1,751.7070",< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),"2,636.3400",< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),"3,164.3167",< .001,< .001,Autocorrelación Significativa


### Descomposición multirresolución

La descomposición wavelet de la volatilidad GK mostró una estructura general similar a la observada en los retornos al cuadrado. La tendencia $A_7$ presentó una meseta moderada durante el colapso del precio del crudo de 2014–2016 y una elevación menor durante el conflicto en Ucrania de 2022, mientras que el período de la pandemia de COVID-19 en 2020 concentró la mayor amplitud. La escala anual $D_7$ reprodujo ese patrón, con oscilaciones pequeñas en 2015–2016 y 2022, y una oscilación dominante en 2020.

Los componentes semestral $D_6$ y trimestral $D_4+D_5$ mantuvieron la misma morfología de espejo observada en los retornos al cuadrado, con amplitudes mayores y frecuencias más altas en los niveles inferiores. En estas escalas, el colapso de 2014–2016 y el conflicto en Ucrania de 2022 también se manifestaron, aunque con amplitudes menores que las de la pandemia de 2020. El período de la pandemia concentró el paquete oscilatorio más agudo. Esta similitud indicó que el estimador GK, a pesar de incorporar información intradía, no alteró de forma sustancial la estructura multiescala de la varianza. Ambos estimadores reflejaron la misma dinámica latente, dominada por los shocks de 2020.

In [46]:
df_plot = df_train.dropna(subset=["gk_daily"]).copy()
gk_vals = df_plot["gk_daily"].values

wavelet = "db4"
level = 7
coeffsgk = pywt.wavedec(gk_vals, wavelet, level=level)

_, _ = tst.plot_wavelet_decomposition(
    coeffs=coeffsgk, index=df_plot.index, n_obs=len(gk_vals),
    wavelet=wavelet, levels=(0, 1, 2, (3, 4)),
    panel_titles=(
        r"Tendencia",
        r"Estacionalidad anual",
        r"Estacionalidad semestral",
        r"Estacionalidad trimestral",
    ),
    ylab=("Volatilidad GK", "Desviación", "Desviación", "Desviación"), xlab="Fecha",
    title_loc="center", right_yaxis=True
)

El componente irregular de la volatilidad GK mostró una reducción considerable del ruido de alta frecuencia respecto a los retornos al cuadrado. Los picos asociados al colapso de 2014–2016 y al conflicto en Ucrania de 2022, que aún se distinguían en $r^2$, se atenuaron de forma notable en esta escala, aunque no desaparecieron por completo. El período de la pandemia de COVID-19 en 2020 mantuvo el impulso más visible, con una amplitud menor que en los retornos al cuadrado. Este comportamiento sugirió que la mayor eficiencia del estimador GK, al incorporar información intradía, mitigó parte de las fluctuaciones de alta frecuencia, pero preservó la señal de los eventos más extremos.

In [48]:
_, _ = tst.plot_wavelet_decomposition(
    coeffsgk, index=df_plot.index, n_obs=len(gk_vals),
    wavelet=wavelet, levels=((5, 6, 7),),
    panel_titles=("Variaciones irregulares",),
    ylab="Desviación", xlab="Fecha",
    title_loc="center",
    ylim=(-0.1, 0.1),
)

### Suavizado por media móvil

El suavizado de la volatilidad GK mostró un comportamiento similar al reportado para los retornos al cuadrado. La ventana de 5 días redujo la mayor parte del ruido de alta frecuencia y reveló la estructura persistente de la varianza. Las ventanas de 10 y 21 días suavizaron aún más la serie y confirmaron los mismos regímenes de volatilidad elevada: el colapso de 2014–2016, la pandemia de COVID-19 en 2020 y el conflicto en Ucrania en 2022. Los niveles de volatilidad suavizada fueron menores que los observados en los retornos al cuadrado, y la señal de fondo apareció más definida, en línea con la mayor eficiencia del estimador GK.

In [23]:
df_plot = df_train.dropna(subset=["gk_daily"]).copy()

_, _ = tst.plot_rolling_smoothing_panels(
    df_plot["gk_daily"],
    titles=(
        "Volatilidad GK",
        "Volatilidad GK (suavizado 5 días)",
        "Volatilidad GK (suavizado 10 días)",
        "Volatilidad GK (suavizado 21 días)"
    ),
    ylab="Volatilidad GK", xlab="Fecha",
    ylim=(None, (-0.001, 0.032), (-0.001, 0.02), (-0.001, 0.02)),
    y_format=("%.3f", None, None, None),
    yticks=(None, np.arange(0.002, 0.032, 0.006), None, None),
    tick_size=10, hspace=0.30
)

## Correlación entre estimadores de volatilidad

Esta sección examina la dependencia entre los retornos al cuadrado y la volatilidad GK. El objetivo es determinar si, pese a la mayor eficiencia teórica del estimador GK, los retornos al cuadrado resultan adecuados para estudiar la volatilidad del Brent, en particular al aplicar ventanas de suavizado.

La correlación contemporánea entre ambas medidas aumentó al ampliar la ventana de suavizado. Este patrón fue consistente con la reducción del ruido de alta frecuencia por parte del filtrado de la variabilidad de corto plazo; la correlación entre los estimadores se incrementó y reveló una estructura común de volatilidad persistente. La relación entre las series mostró un carácter predominantemente lineal, con una asociación monótona que se fortaleció en ventanas largas.

In [24]:
df_corr = df_train[["squared_returns", "gk_daily"]].dropna()

_ = tst.correlation_windows_table(
    df_corr["squared_returns"],
    df_corr["gk_daily"],
    windows=(5, 10, 21),
    window_labels=("Sin suavizar", "5 días", "10 días", "21 días"),
    column_titles=("Ventana", "Pearson", "Spearman")
)

Ventana,Pearson,Spearman
Sin suavizar,0.681***,0.560***
5 días,0.820***,0.821***
10 días,0.879***,0.893***
21 días,0.929***,0.932***


La función de correlación cruzada (*cross-correlation function*, CCF) extendió el análisis de correlación global para examinar la estructura de retardos y contrastar la hipótesis de contemporaneidad entre ambas medidas de volatilidad. Sin suavizar, la correlación máxima se concentró en los retardos $-1$ y $0$, con valores cercanos a $0.7$. El resto de retardos permaneció bajo y dominado por ruido de alta frecuencia. Con la ventana de 5 días, la correlación en todos los retardos aumentó y alcanzó su máximo ($\approx 0.80$) en el retardo $-1$. La correlación en retardos lejanos, como $-12$ y $8$, también mostró un incremento, aunque en menor medida.

:::{admonition} Información: Función de correlación cruzada (CCF)
:class: info

La CCF mide la correlación entre dos series temporales desplazadas en el tiempo. Cada retardo $k$ cuantifica el grado de asociación entre una serie en el momento $t$ y la otra en $t+k$. La expresión del coeficiente de correlación cruzada muestral en términos de covarianza y varianza es:

$$
\rho_{xy}(k) = \frac{\mathrm{Cov}(x_t, y_{t+k})}{\sqrt{\mathrm{Var}(x_t) \cdot \mathrm{Var}(y_t)}}
$$

donde $\mathrm{Cov}(x_t, y_{t+k})$ denota la covarianza cruzada entre $x_t$ y $y_{t+k}$, y $\mathrm{Var}(x_t)$, $\mathrm{Var}(y_t)$ son las varianzas de cada serie calculadas sobre toda la muestra.
:::

Al ampliar la ventana a 10 y 21 días, la curva de correlación se suavizó. La correlación en retardos cercanos a cero superó 0.90, mientras el decaimiento de la dependencia se volvió menos pronunciado, en especial en los retardos negativos. 

In [25]:
df_corr = df_train[["squared_returns", "gk_daily"]].dropna()

_, _ = tst.plot_ccf_panels(
    df_corr["squared_returns"], df_corr["gk_daily"],
    windows=(5, 10, 21), max_lag=20,
    titles=(
        "Sin suavizar",
        "Suavizado 5 días",
        "Suavizado 10 días",
        "Suavizado 21 días"
    ),
    ylab="Cross-correlation", xlab="Retardo",
    ylim=(-0.001, 1.00), y_step=0.1
)

Los resultados del análisis de dependencia entre series sugieren que los retornos al cuadrado constituyen una aproximación adecuada de la volatilidad diaria del Brent frente al estimador GK, en particular al aplicar una ventana de suavizado. El suavizado redujo el ruido de alta frecuencia y reveló la estructura común de volatilidad. La correlación en retardos distintos de cero aumentó con la amplitud de la ventana, en especial en los retardos negativos, como consecuencia de la autocorrelación introducida por el filtrado en cada serie. Una ventana de una semana bursátil (5 días) fue suficiente para filtrar la mayor parte del ruido y mantuvo ese efecto en un nivel reducido.